In [10]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from time import perf_counter

In [11]:
# =========================================================
# 0) Utils: Torch Dice / IoU (works with logits)
# =========================================================

@torch.no_grad()
def dice_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    """
    logits:  [B, 1, H, W] (raw, no sigmoid)
    targets: [B, 1, H, W] float {0,1}
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dims)
    denom = preds.sum(dims) + targets.sum(dims)
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean()

@torch.no_grad()
def iou_from_logits(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    dims = (1, 2, 3)
    intersection = (preds * targets).sum(dims)
    union = preds.sum(dims) + targets.sum(dims) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, device: str = "cuda") -> dict:
    model.eval()
    criterion = nn.BCEWithLogitsLoss()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    n = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device).float().unsqueeze(1)  # [B,1,H,W]

        logits = model(images)  # [B,1,h,w]
        logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

        loss = criterion(logits, masks).item()
        d = dice_from_logits(logits, masks).item()
        j = iou_from_logits(logits, masks).item()

        total_loss += loss
        total_dice += d
        total_iou += j
        n += 1

    if n == 0:
        return {"loss": 0.0, "dice": 0.0, "iou": 0.0}

    return {"loss": total_loss / n, "dice": total_dice / n, "iou": total_iou / n}

In [12]:

# ============================================================
# MaxSigLayerNormalized (with proper C_in -> C_out projection)
# ============================================================
class MaxSigLayerNormalized(nn.Module):
    """
    Args:
        ch_in (int): input channels
        ch_out (int): output channels
        kernel_size (int): odd recommended
        stride (int): keep 1 to preserve H,W
        use_sigmoid_input (bool): apply sigmoid(x) before unfolding
    """
    def __init__(
        self,
        ch_in: int,
        ch_out: int,
        kernel_size: int,
        stride: int = 1,
        use_sigmoid_input: bool = True,
    ):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in = ch_in
        self.ch_out = ch_out
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_sigmoid_input = use_sigmoid_input

        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        # learnable patch weights / bias (shared)
        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))

        # learnable scalars
        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))

        # proper projection
        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"

        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(
            x_in,
            kernel_size=self.kernel_size,
            padding=self.padding,
            stride=self.stride,
        )  # (B, C_in*K*K, L)

        _, CK2, L = x_unfold.shape
        assert CK2 == C_in * self.kernel_area

        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)  # (1,1,K*K,1,1)
        b = torch.sigmoid(self.bias)    # (1,1,K*K,1,1)

        D = torch.maximum(w, x_unfold) + b  # (B,C,K*K,out_h,out_w)

        mean_stat = torch.mean(D, dim=2)         # (B,C,out_h,out_w)
        median_stat, _ = torch.median(D, dim=2)  # (B,C,out_h,out_w)

        center_values = x_unfold[:, :, self.center_index, :, :]  # (B,C,out_h,out_w)

        out = (
            self.weight_center * center_values
            + self.weight_median * median_stat
            - mean_stat
        )  # (B,C_in,out_h,out_w)

        out = self.proj(out)  # (B,C_out,out_h,out_w)
        return out


In [13]:
# =========================================================
# 1) ConvNeXt Building Blocks
# =========================================================

class LayerNorm(nn.Module):
    """
    ConvNeXt-style LayerNorm supporting:
      - channels_last  : (N, H, W, C)
      - channels_first : (N, C, H, W)
    """
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        if data_format not in ("channels_last", "channels_first"):
            raise ValueError("data_format must be 'channels_last' or 'channels_first'")
        self.data_format = data_format
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == "channels_last":
            # x: (N, H, W, C)
            mean = x.mean(-1, keepdim=True)
            var = (x - mean).pow(2).mean(-1, keepdim=True)
            x = (x - mean) / torch.sqrt(var + self.eps)
            x = x * self.weight + self.bias
            return x
        else:
            # x: (N, C, H, W) -> normalize over C
            mean = x.mean(1, keepdim=True)
            var = (x - mean).pow(2).mean(1, keepdim=True)
            x = (x - mean) / torch.sqrt(var + self.eps)
            x = x * self.weight[:, None, None] + self.bias[:, None, None]
            return x

class DropPath(nn.Module):
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0., layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format="channels_last")
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim), requires_grad=True) \
            if layer_scale_init_value > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        residual = x
        x = self.dwconv(x)              # NCHW
        x = x.permute(0, 2, 3, 1)       # NHWC
        x = self.norm(x)                # LN on channels (last dim)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)       # back to NCHW
        x = residual + self.drop_path(x)
        return x


# =========================================================
# 2) ConvNeXt Backbone (Multi-scale Output)
# =========================================================
#Apply it to deep features only (stage 3 or stage 2) using a bottleneck “adapter”
class MSLNAdapter(nn.Module):
    def __init__(self, dim, bottleneck_ratio=4, kernel_size=5, use_sigmoid_input=False):
        super().__init__()
        b = max(8, dim // bottleneck_ratio)
        self.reduce = nn.Conv2d(dim, b, kernel_size=1, bias=False)
        self.msln   = MaxSigLayerNormalized(b, b, kernel_size=kernel_size, stride=1, use_sigmoid_input=use_sigmoid_input)
        self.expand = nn.Conv2d(b, dim, kernel_size=1, bias=False)
        # start at 0 so it behaves like identity at init (very stable)
        self.alpha  = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return x + self.alpha * self.expand(self.msln(self.reduce(x)))
        
class ConvNeXtBackbone(nn.Module):
    def __init__(self, in_chans=1, depths=[3, 3, 9, 3], dims=[96, 192, 384, 768],
                 drop_path_rate=0., layer_scale_init_value=1e-6):
        super().__init__()

        # Downsample layers operate in NCHW, so use channels_first norm
        self.downsample_layers = nn.ModuleList()
        stem = nn.Sequential(
            nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
            LayerNorm(dims[0], eps=1e-6, data_format="channels_first"),
        )
        self.downsample_layers.append(stem)

        for i in range(1, 4):
            down = nn.Sequential(
                LayerNorm(dims[i - 1], eps=1e-6, data_format="channels_first"),
                nn.Conv2d(dims[i - 1], dims[i], kernel_size=2, stride=2),
            )
            self.downsample_layers.append(down)

        self.stages = nn.ModuleList()
        dp_rates = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i in range(4):
            blocks = nn.Sequential(*[
                ConvNeXtBlock(
                    dim=dims[i],
                    drop_path=dp_rates[cur + j],
                    layer_scale_init_value=layer_scale_init_value
                )
                for j in range(depths[i])
            ])
            self.stages.append(blocks)
            cur += depths[i]

        # Output norms also see NCHW, so channels_first
        self.out_norms = nn.ModuleList([LayerNorm(dims[i], eps=1e-6, data_format="channels_first") for i in range(4)])
        
        self.adapters = nn.ModuleList([
            nn.Identity(),                      # stage0
            #MSLNAdapter(dims[0], kernel_size=5),
            
            nn.Identity(),                      # stage1
            #MSLNAdapter(dims[1], kernel_size=5),
            
            #nn.Identity(),                      # stage2
            MSLNAdapter(dims[2], kernel_size=5),
            
            nn.Identity()                       # stage3 only
            #MSLNAdapter(dims[3], kernel_size=5)  
        ])    

    def forward(self, x):
        outs = []
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
            x = self.adapters[i](x) 
            outs.append(self.out_norms[i](x))
        return outs


# =========================================================
# 3) UPerNet Decoder (1-Channel Output)
# =========================================================

class UPerNetDecoder(nn.Module):
    def __init__(self, in_channels=[96, 192, 384, 768], channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(in_ch, channels, 1) for in_ch in in_channels])
        self.fpn_convs = nn.ModuleList([nn.Conv2d(channels, channels, 3, padding=1) for _ in in_channels])

        self.psp_pooling = nn.AdaptiveAvgPool2d(1)
        self.psp_conv = nn.Conv2d(in_channels[-1], channels, 1)

        self.fusion = nn.Conv2d(channels, channels, 3, padding=1)
        self.cls = nn.Conv2d(channels, 1, 1)

    def forward(self, features):
    # Laterals
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        # Top-down pathway: propagate high-level info to lower levels (sizes stay different per level)
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i - 1] = laterals[i - 1] + F.interpolate(
                laterals[i],
                size=laterals[i - 1].shape[2:],
                mode="bilinear",
                align_corners=False
            )

        # FPN conv on each level (still different sizes)
        fpn_outs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]

        # Choose the target (highest) resolution: level 0
        target_size = fpn_outs[0].shape[2:]

        # Upsample ALL FPN outputs to the same size so we can sum/average
        fpn_outs = [
            o if o.shape[2:] == target_size else F.interpolate(o, size=target_size, mode="bilinear", align_corners=False)
            for o in fpn_outs
        ]

        # PSP from the deepest feature, then upsample to target_size
        psp = self.psp_conv(self.psp_pooling(features[-1]))
        psp = F.interpolate(psp, size=target_size, mode="bilinear", align_corners=False)

        # Fusion (now shapes match)
        fused = self.fusion(sum(fpn_outs) / len(fpn_outs) + psp)
        return self.cls(fused)


# =========================================================
# 4) Full Segmentation Model
# =========================================================

class ConvNeXtBinarySegmentation(nn.Module):
    def __init__(self, backbone="tiny"):
        super().__init__()
        if backbone == "tiny":
            self.backbone = ConvNeXtBackbone(depths=[3, 3, 9, 3], dims=[96, 192, 384, 768])
            in_channels = [96, 192, 384, 768]
        elif backbone == "small":
            self.backbone = ConvNeXtBackbone(depths=[3, 3, 27, 3], dims=[96, 192, 384, 768])
            in_channels = [96, 192, 384, 768]
        elif backbone == "base":
            self.backbone = ConvNeXtBackbone(depths=[3, 3, 27, 3], dims=[128, 256, 512, 1024])
            in_channels = [128, 256, 512, 1024]
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        self.decoder = UPerNetDecoder(in_channels=in_channels)

    def forward(self, x):
        features = self.backbone(x)
        return self.decoder(features)

In [14]:
from glob import glob
class BinarySegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, augment=False):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.augment = augment        
        self.mask_names = glob(str(mask_dir / "*.bmp"))

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self,index):
        mask_name=self.mask_names[index]
        mask=cv2.imread(mask_name,cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())#ejbari
        mask= (mask>0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)
        mask = torch.from_numpy(mask)

        imgn=str(mask_name).replace('/masks/','/images/').replace('_anno.bmp','.bmp')        

        image=cv2.imread(imgn,cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)#ejbari
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_NEAREST)
        image = np.expand_dims(image, axis=0)
        image = torch.from_numpy(image)

        return image, mask


In [15]:
# =========================================================
# 6) Training Function (NO test_loader here)
# =========================================================

def train_model(model, train_loader, val_loader, epochs=50, device="cuda", save_dir="checkpoints"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": 1e-4},
            {"params": model.decoder.parameters(), "lr": 1e-4},
        ],
        weight_decay=0.05,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_dice = -1.0

    for epoch in range(epochs):
        start_time = perf_counter()
        model.train()
        train_loss = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device).float().unsqueeze(1)  # [B,1,H,W]

            optimizer.zero_grad()
            logits = model(images)  # [B,1,h,w]
            logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= max(1, len(train_loader))
        scheduler.step()

        # Validation
        val_stats = evaluate_model(model, val_loader, device=device)
        val_loss = val_stats["loss"]
        val_dice = val_stats["dice"]
        val_iou = val_stats["iou"]

        print(
            f"Epoch {epoch+1:>3}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}"
        )

        # Save best model by Dice
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                    "val_dice": val_dice,
                    "val_iou": val_iou,
                },
                save_dir / "best_model_convnext_maxsig.pth",
            )
            print(f"✓ Saved best model (Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f})")
        end_time = perf_counter()
        print(f"Time taken: {end_time - start_time:.2f} seconds.\n")

    return model

In [16]:
# =========================================================
# 7) Inference Function
# =========================================================

@torch.no_grad()
def predict(model, image_path, device="cuda", threshold=0.5, img_size=512):
    model.eval()
    model = model.to(device)

    image_path = str(image_path)
    img = cv2.imread(image_path, cv2.IMREAD_ANYDEPTH)
    img = (img-img.min())/(img.max()-img.min()).float()
    #if img is None:
    #    raise RuntimeError(f"cv2.imread failed for image: {image_path}")

    orig_h, orig_w = img.shape[:2]    
    img_resized = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

    x = torch.from_numpy(img_resized).unsqueeze(0)
    x = x.to(device)

    logits = model(x)  # [1,1,h,w]
    probs = torch.sigmoid(logits)
    pred = (probs > threshold).float().squeeze(0).squeeze(0).cpu().numpy()

    pred_resized = cv2.resize(pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    prob_resized = cv2.resize(probs.squeeze(0).squeeze(0).cpu().numpy(), (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    return pred_resized, prob_resized

In [17]:
# =========================================================
# 8) Visualization Function
# =========================================================

def visualize_prediction(image_path, pred_mask, output_path="output.png"):
    image_path = str(image_path)
    output_path = str(output_path)

    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"cv2.imread failed for image: {image_path}")

    mask_color = np.zeros_like(img)
    mask_color[pred_mask == 1] = [0, 255, 0]  # green overlay in BGR

    overlay = cv2.addWeighted(img, 0.7, mask_color, 0.3, 0)
    cv2.imwrite(output_path, overlay)
    print(f"✓ Saved visualization to {output_path}")

In [18]:
# =========================================================
# 9) Main
# =========================================================

if __name__ == "__main__":
    torch.backends.cudnn.benchmark = True

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    IMAGE_SIZE = 512
    BATCH_SIZE = 8
    EPOCHS = 100
    BACKBONE = "tiny"

    ROOT_DIR = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/BUSI_WHU/dataset')

    print(f"Using device: {DEVICE}")

    model = ConvNeXtBinarySegmentation(backbone=BACKBONE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    train_dataset = BinarySegmentationDataset(
        image_dir=ROOT_DIR / "train" / "images",
        mask_dir=ROOT_DIR / "train" / "masks",
        img_size=IMAGE_SIZE,
        augment=False,
    )

    val_dataset = BinarySegmentationDataset(
        image_dir=ROOT_DIR / "val" / "images",
        mask_dir=ROOT_DIR / "val" / "masks",
        img_size=IMAGE_SIZE,
        augment=False,
    )

    test_dataset = BinarySegmentationDataset(
        image_dir=ROOT_DIR / "test" / "images",
        mask_dir=ROOT_DIR / "test" / "masks",
        img_size=IMAGE_SIZE,
        augment=False,
    )

    # Windows tip: if you hit DataLoader worker issues, set num_workers=0
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda"))
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE == "cuda"))
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE == "cuda"))

    print(f"\nTrain dataset size: {len(train_dataset)}")
    print(f"Train loader batches: {len(train_loader)}\n")
    data_batch,mask_batch = next(iter(train_loader))
    print("data_batch.size=",data_batch.size()) 
    
    print(f"\nValidation dataset size: {len(val_dataset)}")
    print(f"Validation loader batches: {len(val_loader)}\n")
    
    print(f"Test dataset size: {len(test_dataset)}") 
    print(f"Test loader batches: {len(test_loader)}\n")

    save_dir = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/BUSI_WHU/'

    # ---------------------------------------------------------
    # train_model takes (model, train_loader, val_loader, ...)
    # ---------------------------------------------------------
    print("Training started...\n")
    model = train_model(
        model,
        train_loader,
        val_loader,
        epochs=EPOCHS,
        device=DEVICE,
        save_dir=save_dir
    )

    # Load best model
    ckpt_path = save_dir +"best_model_convnext_maxsig.pth"
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"\n✓ Loaded best model from epoch {checkpoint['epoch'] + 1} | Val Dice: {checkpoint['val_dice']:.4f}")

    # Evaluate on test set
    test_stats = evaluate_model(model.to(DEVICE), test_loader, device=DEVICE)
    print(f"TEST | Loss: {test_stats['loss']:.4f} | Dice: {test_stats['dice']:.4f} | IoU: {test_stats['iou']:.4f}")
    print("Finished.")

    # Example inference on first test image (robust to extensions)
    #sample_img_path = test_dataset.images[0]
    #pred_mask, prob_map = predict(model, sample_img_path, device=DEVICE, threshold=0.5, img_size=IMAGE_SIZE)

    #out_vis = ROOT_DIR / "segmentation_output.png"
    #visualize_prediction(sample_img_path, pred_mask, out_vis)

Using device: cuda
Total parameters: 31,418,582

Train dataset size: 555
Train loader batches: 70

data_batch.size= torch.Size([8, 1, 256, 256])

Validation dataset size: 186
Validation loader batches: 24

Test dataset size: 186
Test loader batches: 24

Training started...

Epoch   1/100 | Train Loss: 0.2540 | Val Loss: 0.2088 | Val Dice: 0.0001 | Val IoU: 0.0001
✓ Saved best model (Val Dice: 0.0001 | Val IoU: 0.0001)
Time taken: 20.19 seconds.

Epoch   2/100 | Train Loss: 0.1482 | Val Loss: 0.1405 | Val Dice: 0.5620 | Val IoU: 0.4309
✓ Saved best model (Val Dice: 0.5620 | Val IoU: 0.4309)
Time taken: 18.13 seconds.

Epoch   3/100 | Train Loss: 0.1263 | Val Loss: 0.1279 | Val Dice: 0.5226 | Val IoU: 0.4014
Time taken: 11.76 seconds.

Epoch   4/100 | Train Loss: 0.1171 | Val Loss: 0.1177 | Val Dice: 0.5635 | Val IoU: 0.4383
✓ Saved best model (Val Dice: 0.5635 | Val IoU: 0.4383)
Time taken: 14.65 seconds.

Epoch   5/100 | Train Loss: 0.1142 | Val Loss: 0.1170 | Val Dice: 0.4828 | Val Io